In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

#df = pd.read_csv(PROCESSED_DIR / "dawn_revenue_combined_clean.csv")
#df.head()

# Load the CSV file into a pandas DataFrame
candidate_paths = [
    RAW_DIR / "dawn_revenue_accounts_combined_history.csv",
    PROJECT_ROOT.parent / "data" / "dawn_revenue_accounts_combined_history.csv",
    PROJECT_ROOT.parent / "dawn_aggregate" / "data" / "dawn_revenue_accounts_combined_history.csv",
    PROCESSED_DIR / "dawn_revenue_accounts_combined_history.csv",
]

df_all = None
for path in candidate_paths:
    if path.exists():
        df_all = pd.read_csv(path)
        break

if df_all is None:
    raise FileNotFoundError(
        "Could not find 'dawn_revenue_accounts_combined_history.csv'. "
        f"Checked: {[str(p) for p in candidate_paths]}"
    )
df_all.head()

,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
0,CR 010 00008235185,7/29/2019,200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
1,CR 010 00008235187,7/30/2019,10724.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
2,CR 010 070619ZUFFA,7/30/2019,363822.43,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
3,CR 010 072519MGMTG,7/30/2019,421738.50,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
4,CR 010 00008235189,8/12/2019,752.80,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020


In [2]:
print(df_all['fy'].unique())

[2020 2021 2022 2023 2024 2025 2026 2027 2002 2003 2004 2005 2006 2007
 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2000 2001]


In [3]:
# Removing fy 2027 as it is not a complete year and may skew the analysis

df_all = df_all[df_all['fy'] != 2027]
print(f"Shape of DataFrame after dropping FY2027 data: {df_all.shape}")
display(df_all.head())

Shape of DataFrame after dropping FY2027 data: (332189, 10)


,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
0,CR 010 00008235185,7/29/2019,200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
1,CR 010 00008235187,7/30/2019,10724.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
2,CR 010 070619ZUFFA,7/30/2019,363822.43,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
3,CR 010 072519MGMTG,7/30/2019,421738.50,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
4,CR 010 00008235189,8/12/2019,752.80,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020


In [4]:
# Dropping GL Number 42 as it is not relevant to the analysis

df_all_filtered = df_all[df_all['GL Number'] != 42]
print(f"Shape of DataFrame after dropping GL Number 42: {df_all_filtered.shape}")
display(df_all_filtered.head())

Shape of DataFrame after dropping GL Number 42: (326516, 10)


,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
0,CR 010 00008235185,7/29/2019,200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
1,CR 010 00008235187,7/30/2019,10724.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
2,CR 010 070619ZUFFA,7/30/2019,363822.43,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
3,CR 010 072519MGMTG,7/30/2019,421738.50,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
4,CR 010 00008235189,8/12/2019,752.80,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020


In [5]:
# Convert Amount to numeric first
df_all['Amount'] = pd.to_numeric(df_all['Amount'], errors='coerce')

# Keep everything except GL 42 when Amount is greater than 0
df_all_filtered = df_all[
    ~(
        (df_all['GL Number'] == 42) &
        (df_all['Amount'] > 0)
    )
].copy()

print(f"Shape after dropping only GL 42 with Amount > 0: {df_all_filtered.shape}")
display(df_all_filtered.head())

Shape after dropping only GL 42 with Amount > 0: (326708, 10)


,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
0,CR 010 00008235185,7/29/2019,200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
1,CR 010 00008235187,7/30/2019,10724.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
2,CR 010 070619ZUFFA,7/30/2019,363822.43,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
3,CR 010 072519MGMTG,7/30/2019,421738.50,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
4,CR 010 00008235189,8/12/2019,752.80,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020


## General Fund Calculation

In [6]:
# filtering the DataFrame for BA Number 9999 and specific GL Numbers that we require for the analysis. These GL Numbers are related to the General Fund and are relevant to our analysis.

gl_numbers_to_keep = [3107, 3114, 3115, 3120, 3202, 3203, 3204, 3220, 3240, 3255, 3271, 3274, 3276, 3291, 4102, 4403, 4408, 4409]
df_filtered_ba_gl = df_all_filtered[(df_all_filtered['BA Number'] == 9999) & (df_all_filtered['GL Number'].isin(gl_numbers_to_keep))]
display(df_filtered_ba_gl.head())
print(f"Shape of DataFrame after filtering for BA Number 9999 and specific GL Numbers: {df_filtered_ba_gl.shape}")

,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
268244,CR 707 610684,1/6/2000,1114.5,2004,2653,3107,"MISC LICENSES, FEES, PERMITS",Minor,9999,2000
268245,CR 741 610838,1/7/2000,100.0,2004,2653,3107,"MISC LICENSES, FEES, PERMITS",Minor,9999,2000
268246,CR 741 610839,1/7/2000,100.0,2004,2653,3107,"MISC LICENSES, FEES, PERMITS",Minor,9999,2000
268247,CR 089 747883,1/13/2000,234.0,2004,2653,3107,"MISC LICENSES, FEES, PERMITS",Minor,9999,2000
268248,CR 707 610686,1/13/2000,1282.5,2004,2653,3107,"MISC LICENSES, FEES, PERMITS",Minor,9999,2000


Shape of DataFrame after filtering for BA Number 9999 and specific GL Numbers: (39722, 10)


In [7]:
# filtering the DataFrame for BA Number 9999 and specific GL Numbers that we require for the analysis. These GL Numbers are related to the General Fund and are relevant to our analysis.

gl_numbers_to_keep = [3107, 3114, 3115, 3120, 3202, 3203, 3204, 3220, 3240, 3255, 3271, 3274, 3276, 3291, 4102, 4403, 4408, 4409]

mask = (
    (df_all_filtered['BA Number'] != 9999) |
    (
        (df_all_filtered['BA Number'] == 9999) &
        (df_all_filtered['GL Number'].isin(gl_numbers_to_keep))
    )
)

df_filtered = df_all_filtered.loc[mask].copy()

display(df_filtered.head())
print(f"Shape after keeping all BA values and limiting only BA 9999 GL codes: {df_filtered.shape}")

,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
0,CR 010 00008235185,7/29/2019,200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
1,CR 010 00008235187,7/30/2019,10724.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
2,CR 010 070619ZUFFA,7/30/2019,363822.43,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
3,CR 010 072519MGMTG,7/30/2019,421738.50,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
4,CR 010 00008235189,8/12/2019,752.80,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020


Shape after keeping all BA values and limiting only BA 9999 GL codes: (295252, 10)


In [8]:
# Now I want to keep only the rows where the 'BA Nnumber' column is 9010, 9040, 9050, 9090, 9130, 9360, 9600, 9705, 9707, 9741, 9748, 9752, 9999.    
# I will filter the DataFrame accordingly.
ba_numbers_to_keep = [9010, 9040, 9050, 9090, 9130, 9360, 9600, 9705, 9707, 9741, 9748, 9752, 9999]

df_filtered_ba = df_filtered[df_filtered['BA Number'].isin(ba_numbers_to_keep)]
display(df_filtered_ba.head())
print(f"Shape of DataFrame after filtering for specific BA Numbers: {df_filtered_ba.shape}")


,Doc Number,Date,Amount,level_1_id,level_2_id,GL Number,description,Type,BA Number,fy
0,CR 010 00008235185,7/29/2019,200.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
1,CR 010 00008235187,7/30/2019,10724.00,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
2,CR 010 070619ZUFFA,7/30/2019,363822.43,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
3,CR 010 072519MGMTG,7/30/2019,421738.50,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020
4,CR 010 00008235189,8/12/2019,752.80,25842,70021,3102,ATHLETIC COMM GATE/TV FEES,Minor,9010,2020


Shape of DataFrame after filtering for specific BA Numbers: (295252, 10)


In [9]:
import plotly.express as px

# Group by fiscal year and sum the amount
#df_gf = df_filtered_ba.groupby('fy')['Amount'].sum().reset_index()



df_gf = (
    df_filtered_ba.assign(Amount=pd.to_numeric(df_filtered_ba['Amount'], errors='coerce'))
    .groupby('fy')['Amount']
    .sum()
    .reset_index()
)


# Create an interactive line plot
fig = px.line(
    df_gf,
    x='fy',
    y='Amount',
    title='Total Amount by Fiscal Year',
    labels={'fy': 'Fiscal Year', 'Amount': 'Total Amount'}
)

fig.update_traces(mode='lines+markers')
fig.update_layout(
    hovermode='x unified',
    xaxis_title='Fiscal Year',
    yaxis_title='Total Amount',
    xaxis=dict(tickmode='linear', dtick=1) # Ensure yearly ticks
)

fig.show()

In [10]:
df_gf_table = df_filtered_ba.groupby('fy')['Amount'].sum().reset_index()
df_gf_table.columns = ['Fiscal Year', 'Total Amount']

pd.options.display.float_format = '{:,.2f}'.format
display(df_gf_table)
pd.options.display.float_format = None # Reset to default for other outputs

,Fiscal Year,Total Amount
0,2000,"12,633,446.84"
1,2001,"17,798,449.05"
2,2002,"1,758,431,755.10"
3,2003,"1,815,046,682.10"
4,2004,"2,393,262,682.67"
5,2005,"2,730,971,122.15"
6,2006,"3,044,113,849.54"
7,2007,"3,101,473,360.61"
8,2008,"2,987,809,976.33"
9,2009,"2,669,801,183.03"


In [11]:
df_2026 = df_filtered_ba[df_filtered_ba['fy'] == 2026]
df_2026_by_ba_gl = df_2026.groupby(['BA Number', 'GL Number'])['Amount'].sum().reset_index()
df_2026_by_ba_gl.columns = ['BA Number', 'GL Number', 'Total Amount']

# Format the 'Total Amount' column for better readability
pd.options.display.float_format = '{:,.2f}'.format
display(df_2026_by_ba_gl)
pd.options.display.float_format = None

,BA Number,GL Number,Total Amount
0,9010,3102,"5,997,486.03"
1,9010,3103,"165,062.10"
2,9010,3271,"22,936.62"
3,9040,3105,"3,849,325.00"
4,9040,3113,"127,042,266.00"
...,...,...,...
95,9999,3291,"643,757.92"
96,9999,4102,"3,000,000.00"
97,9999,4403,"20,670.03"
98,9999,4408,"1,036,940.68"


In [12]:
df_2026_table = (
    df_2026.groupby(['BA Number', 'GL Number'])['Amount']
    .sum()
    .sort_index()
)

pd.options.display.float_format = '{:,.2f}'.format
display(df_2026_table)
pd.options.display.float_format = None

BA Number  GL Number
9010       3102          5,997,486.03
           3103            165,062.10
           3271             22,936.62
9040       3105          3,849,325.00
           3113        127,042,266.00
                            ...      
9999       3291            643,757.92
           4102          3,000,000.00
           4403             20,670.03
           4408          1,036,940.68
           4409            125,000.00
Name: Amount, Length: 100, dtype: float64

In [13]:
import pandas as pd
import plotly.express as px

ba_numbers_to_keep = [9010, 9040, 9050, 9090, 9130, 9360, 9600, 9705, 9707, 9741, 9748, 9752, 9999]

df_2026 = df_filtered[
    (df_filtered['fy'] == 2026) &
    (df_filtered['BA Number'].isin(ba_numbers_to_keep))
].copy()

df_2026['Amount'] = pd.to_numeric(df_2026['Amount'], errors='coerce')

plot_df = (
    df_2026.groupby(['BA Number', 'GL Number'], as_index=False)['Amount']
    .sum()
    .sort_values(['BA Number', 'GL Number'])
)

# One interactive chart per BA with values shown on each bar
for ba in sorted(plot_df['BA Number'].unique()):
    ba_df = plot_df[plot_df['BA Number'] == ba].copy()

    fig = px.bar(
        ba_df,
        x='GL Number',
        y='Amount',
        title=f'FY 2026 Total Amount by GL for BA {ba}',
        labels={'GL Number': 'GL Number', 'Amount': 'Total Amount'},
        color='Amount',
        color_continuous_scale='Viridis',
        text='Amount'
    )

    fig.update_traces(
        texttemplate='%{text:,.2f}',
        textposition='outside',
        cliponaxis=False
    )

    fig.update_layout(
        xaxis=dict(type='category'),
        template='plotly_white',
        hovermode='closest',
        yaxis_title='Total Amount',
        xaxis_title='GL Number'
    )
    fig.show()